# From FNN to RNN — Why Sequence Architecture Matters

## The Problem with FNN on Sequential Data

A FNN takes a flat vector as input. To use past information, you manually build a window of lagged returns:
```python
input = [return_t-5, return_t-4, return_t-3, return_t-2, return_t-1]  →  predict volatility_t
```

This works, but has three fundamental limitations.

---

## Limitation 1 — Order Blindness

The FNN assigns an independent weight to each lag. It has no built-in understanding that lag 1 and lag 2 are neighbors in time, or that one happened before the other. The temporal ordering only exists because you constructed the vector that way.

**Proof:** shuffle the elements of the input vector randomly — the FNN produces nearly identical results. An RNN would break completely.

---

## Limitation 2 — Fixed Hard Cutoff

You must decide upfront how many lags to include. If volatility clustering from 45 days ago is relevant today, a window of 30 days will never capture it — no matter how well you train the model.

---

## Limitation 3 — Parameter Explosion

Every extra lag adds a full set of input weights. With 60 days and 5 features, the first layer already has 300 weights just for the inputs. Doubling the window doubles the parameters.

---

## What the RNN Changes

Instead of consuming a flat vector, the RNN processes one time step at a time and maintains a **hidden state** — a compressed memory of everything seen so far.

The same weight matrix is reused at every step — so the number of parameters does not grow with sequence length.

---

## Why This Matters for Volatility Forecasting

Volatility clustering means high volatility today predicts high volatility tomorrow — and this effect can persist for **weeks or months**. A FNN with a 10-day window is structurally blind to anything beyond day 10.

The RNN hidden state accumulates this context automatically. A spike 3 weeks ago influences h_t-20, which flows into h_t-19, all the way to today's prediction. The network decides what to keep and what to forget — rather than you deciding upfront with a fixed window.

---

## What RNN Does Not Solve — The Vanishing Gradient

In practice the hidden state cannot carry information indefinitely. During backpropagation through many time steps, gradients shrink exponentially — the network effectively forgets anything beyond ~10-20 steps. This is the **vanishing gradient problem**, and it is the motivation for the LSTM.

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.append('src')
from dl_utils.Operation import WeightMultiply, Add
from dl_utils.NumberWithGrad import NumberWithGrad

Custom Implementation

In [ ]:
a = NumberWithGrad(3)
b = a * 4
c = b + 5


With Torch

In [14]:
a = torch.tensor(3.0, requires_grad=True)
b = a * 4
c = b + 3
c.backward()
print(a.grad)

tensor(4.)
